# Time Series Course — m11-e3 (block 3)

_From the **Time Series & Forecasting** course on Zylo._

Run all cells (`Runtime → Run all` or `Ctrl+F9`) to execute.

In [ ]:
# Informer: ProbSparse Attention (conceptual)
class ProbSparseAttention(nn.Module):
    """Only top-u queries compute full attention.
    u = c * ln(L) where c is a sampling factor."""
    def __init__(self, d_model, n_heads, factor=5):
        super().__init__()
        self.factor = factor
        self.d_k = d_model // n_heads

    def forward(self, Q, K, V):
        B, L, H, D = Q.shape
        u = int(self.factor * math.log(L))  # number of dominant queries

        # Measure "sparsity" of each query's attention distribution
        # High KL from uniform = query is selective = worth computing
        scores_sample = torch.matmul(Q, K.transpose(-1, -2))
        M = scores_sample.max(-1).values - scores_sample.mean(-1)
        top_u = M.topk(u, dim=-1).indices  # dominant queries

        # Only compute full attention for top-u queries
        # Rest get mean(V) — a constant approximation
        # This reduces O(L^2) to O(L log L)
        return output